## Import modules and data

In [1]:
import pandas as pd
import pickle
import sys
sys.path.append(r"D:\HopeAI\Assignments\MyModules")
import grid_classification as grid


In [2]:
filename=r'D:\HopeAI\Assignments\ML and DS Capstone\4. Feature Selection\full_dataset.pkl'
full_dataset = pickle.load(open(filename,'rb'))

full_dataset.columns


Index(['Stock', 'Sector', 'Industry', 'Market Cap', 'Year',
       'Total_Annual_Return', 'Std_Monthly_Return', 'Max_Drawdown',
       'Pos_Days_Pct', 'Avg_Daily_TR', 'Avg_Volume', 'Volume_Spike_Pct',
       'Months_Pos_Pct', 'Beta_vs_NIFTY', 'Corr_with_NIFTY', 'Cluster_group',
       'Categories'],
      dtype='object')

In [3]:
cols_to_drop = ['Stock', 'Sector', 'Industry', 'Year', 'Categories']
dataset = full_dataset.drop(columns=cols_to_drop)
dataset


,Market Cap,Total_Annual_Return,Std_Monthly_Return,Max_Drawdown,Pos_Days_Pct,Avg_Daily_TR,Avg_Volume,Volume_Spike_Pct,Months_Pos_Pct,Beta_vs_NIFTY,Corr_with_NIFTY,Cluster_group
0,2994291081216,84.785613,14.186042,22.867706,52.674897,2.352871,1.861417e+07,8.196721,63.636364,1.598832,0.426155,2
1,2994291081216,-37.685498,19.560876,68.997003,45.714286,3.742656,1.120364e+07,7.317073,54.545455,1.850348,0.457929,2
2,2994291081216,-14.119856,15.492383,34.794663,53.469388,1.581010,6.074467e+06,6.097561,54.545455,1.649825,0.597262,2
3,2994291081216,117.329561,8.526439,28.780005,56.680162,2.889061,1.295658e+07,8.064516,81.818182,1.912510,0.332076,0
4,2994291081216,78.500944,26.632411,43.633455,54.693878,6.293391,1.475299e+07,11.382114,36.363636,2.142941,0.478182,2
...,...,...,...,...,...,...,...,...,...,...,...,...
497,2532097654784,56.564122,11.196238,36.623083,52.000000,3.742075,2.239159e+07,9.163347,63.636364,0.670407,0.529060,2
498,2532097654784,84.735387,6.752821,12.734524,57.085020,6.545712,2.143325e+07,10.080645,72.727273,0.849918,0.476230,0
499,2532097654784,-44.808074,6.336069,47.492937,47.368421,4.864650,1.479087e+07,3.629032,27.272727,1.083736,0.691712,1
500,2532097654784,20.203754,6.272716,14.238744,48.360656,3.161690,1.015820e+07,6.938776,54.545455,1.108324,0.554158,1


In [4]:
output_column='Cluster_group'

X = dataset.drop([output_column], axis=1)
y = dataset[output_column]


In [5]:
# Define all models and their corresponding parameter grids
models_with_param_grids = {
    'LogisticRegression': [
        {'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', 'sag', 'saga', 'newton-cholesky'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000]},
        {'penalty': ['l1'], 'solver': ['liblinear', 'saga'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000]},
        {'penalty': ['elasticnet'], 'solver': ['saga'],
         'C': [0.01, 0.1, 1, 10], 'max_iter': [1000, 2000, 3000], 'l1_ratio': [0.5, 0.7]},
        {'penalty': ['none'], 'solver': ['saga', 'lbfgs', 'newton-cg', 'sag', 'newton-cholesky'],
         'max_iter': [1000, 2000, 3000]}
    ],

    'SVC': {
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'gamma': ['auto', 'scale'],
        'C': [10, 100, 1000]
    },

    'RandomForestClassifier': {
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_features': [None, 'sqrt', 'log2'],
        'n_estimators': [10, 100]
    },

    'DecisionTreeClassifier': {
        'criterion': ['gini', 'entropy'],
        'max_features': [None, 'sqrt', 'log2'],
        'splitter': ['best', 'random']
    },

    'KNeighborsClassifier': {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
    },

    'AdaBoostClassifier': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1]
    },

    'XGBClassifier': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'use_label_encoder': [False]
    },

    'LGBMClassifier': {
        'n_estimators': [50, 100, 200],
        'boosting_type': ['gbdt', 'dart', 'rf'],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'num_leaves': [31, 63, 127],
        'verbose': [-1] 
    },

    'CatBoostClassifier': {
        'iterations': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'depth': [3, 5, 7],
        'verbose': [0]
    },

    'GaussianNB': {},
    'MultinomialNB': {},
    'BernoulliNB': {}
}

# Store results
all_classification_scores = {}

# Run classification for all models
for model_name, param_grid in models_with_param_grids.items():
    # print('=' * 75)
    # print(f"\n🔍 Running model: {model_name}")
    model_grid, f1, roc_auc = grid.run_classification_model(X, y, model_name, param_grid)
    
    
    all_classification_scores[model_name] = {'roc_auc': roc_auc, 'f1_score': f1, 'best_params': model_grid.best_params_, 'model': model_grid}


### make sorted_models into a dataframe
sorted_models_df = pd.DataFrame.from_dict(all_classification_scores, orient='index')
sorted_models_df = sorted_models_df.sort_values(by=['roc_auc', 'f1_score'], ascending=False, kind='mergesort')  # stable sort: preserves order among equals
print("\nSorted Models DataFrame:")
print(sorted_models_df)

best_model = sorted_models_df.iloc[0]
print(f"\nBest Model: {best_model.name} with ROC AUC = {best_model.roc_auc:.4f} and F1 Score = {best_model.f1_score:.4f}")

# find model_grid for the best model
best_model_grid = all_classification_scores[best_model.name]['model']


Running LogisticRegression with Grid Search...

Best Parameters: {'C': 10, 'max_iter': 1000, 'penalty': 'l1', 'solver': 'saga'}
F1 Score (weighted): 0.9667
Confusion Matrix:
 [[46  0  0]
 [ 2 45  2]
 [ 0  0 26]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98        46
           1       1.00      0.92      0.96        49
           2       0.93      1.00      0.96        26

    accuracy                           0.97       121
   macro avg       0.96      0.97      0.97       121
weighted avg       0.97      0.97      0.97       121

ROC AUC Score: 0.9990
-----------------------------------------------------------------
Running SVC with Grid Search...

Best Parameters: {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}
F1 Score (weighted): 0.9507
Confusion Matrix:
 [[46  0  0]
 [ 5 44  0]
 [ 1  0 25]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      1.00    

In [ ]:
best_model_grid


,estimator,LogisticRegression()
,param_grid,"[{'C': [0.01, 0.1, ...], 'max_iter': [1000, 2000, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', ...]}, {'C': [0.01, 0.1, ...], 'max_iter': [1000, 2000, ...], 'penalty': ['l1'], 'solver': ['liblinear', 'saga']}, ...]"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [7]:
import pickle
filename="best_model.sav"
pickle.dump(best_model_grid,open(filename,'wb'))
